In [0]:
catalog = "CineData_Analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: CineData_Analytics
bronze_schema: CineData_Analytics.bronze
silver_schema: CineData_Analytics.silver
gold_schema: CineData_Analytics.gold
landing_path: /Volumes/CineData_Analytics/bronze/Landing


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

print("Catalog, schemas e volume prontos.")

Catalog, schemas e volume prontos.


In [0]:
from pyspark.sql.functions import current_timestamp

path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"

df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=True)
df_financials_raw = spark.read.csv(path_financials, header=True, inferSchema=True)
df_info_raw = spark.read.csv(path_info, header=True, inferSchema=True)
df_metrics_raw = spark.read.csv(path_metrics, header=True, inferSchema=True)
df_reviews_raw = spark.read.csv(path_reviews, header=True, inferSchema=True)

In [0]:
# Regra de negócio: Todas as tabelas da camada Bronze devem registrar a data e hora da ingestão para permitir rastreabilidade e auditoria dos dados.
df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_credits_and_tags")

df_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_financials")

df_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_info")

df_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_metrics")

df_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{catalog}.bronze.tb_movies_reviews")


display(spark.table(f"{catalog}.bronze.tb_movies_reviews").limit(5))

id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-16T05:43:47.681Z
637007,Lucas Reis 602,3.9,null,2026-10-16T05:43:47.681Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-16T05:43:47.681Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-16T05:43:47.681Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-16T05:43:47.681Z


In [0]:
# Regra de negócio: A cotação do dólar pode ser utilizada posteriormente para conversão dos valores financeiros dos filmes de USD para BRL.
from pyspark.sql import functions as F

dbutils.widgets.removeAll()

dbutils.widgets.text("data_inicio", "10-09-2026")
dbutils.widgets.text("data_fim", "10-16-2026")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Data início: {data_inicio}")
print(f"Data fim: {data_fim}")

df_cotacao_raw = spark.read \
    .option("multiline", "true") \
    .json(
        f"{landing_path}/cotacao_dolar.json"
    )

df_cotacao = (
    df_cotacao_raw
    .select(
        F.explode("value").alias("cotacao")
    )
    .select(
        F.col("cotacao.dataHoraCotacao").alias(
            "data_hora_cotacao"
        ),
        F.col("cotacao.cotacaoCompra").alias(
            "cotacao_compra"
        )
    )
)

df_cotacao = (
    df_cotacao
    .withColumn(
        "data_cotacao",
        F.to_date("data_hora_cotacao")
    )
)

df_cotacao = (
    df_cotacao
    .filter(
        (
            F.col("data_cotacao")
            >= F.to_date(
                F.lit(data_inicio),
                "MM-dd-yyyy"
            )
        )
        &
        (
            F.col("data_cotacao")
            <= F.to_date(
                F.lit(data_fim),
                "MM-dd-yyyy"
            )
        )
    )
)

df_cotacao = (
    df_cotacao
    .withColumn(
        "ingestion_datetime",
        F.current_timestamp()
    )
    .drop("data_cotacao")
)

display(df_cotacao)

df_cotacao.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{bronze_schema}.tb_cotacao_dolar"
    )

display(
    spark.table(
        f"{bronze_schema}.tb_cotacao_dolar"
    )
)

Data início: 10-09-2026
Data fim: 10-16-2026


data_hora_cotacao,cotacao_compra,ingestion_datetime
2026-10-09 13:07:30.746597,4.9886,2026-10-16T13:38:12.847Z
2026-10-13 13:04:26.866133,5.009,2026-10-16T13:38:12.847Z
2026-10-14 13:05:17.833426,5.0435,2026-10-16T13:38:12.847Z
2026-10-15 13:05:33.305854,5.0515,2026-10-16T13:38:12.847Z


data_hora_cotacao,cotacao_compra,ingestion_datetime
2026-10-09 13:07:30.746597,4.9886,2026-10-16T13:37:02.594Z
2026-10-13 13:04:26.866133,5.009,2026-10-16T13:37:02.594Z
2026-10-14 13:05:17.833426,5.0435,2026-10-16T13:37:02.594Z
2026-10-15 13:05:33.305854,5.0515,2026-10-16T13:37:02.594Z
2026-10-09 13:07:30.746597,4.9886,2026-10-16T13:38:14.028Z
2026-10-13 13:04:26.866133,5.009,2026-10-16T13:38:14.028Z
2026-10-14 13:05:17.833426,5.0435,2026-10-16T13:38:14.028Z
2026-10-15 13:05:33.305854,5.0515,2026-10-16T13:38:14.028Z
2026-10-09 13:07:30.746597,4.9886,2026-10-16T13:36:24.312Z
2026-10-13 13:04:26.866133,5.009,2026-10-16T13:36:24.312Z
